# Chapter 14: Building Custom Assistants

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build a **BA/QA Assistant** -- a multi-tool conversational agent that can analyze requirements, generate test cases, review documents, and answer process questions using function calling.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Completed Chapter 13 lab (RAG concepts)


In [ ]:
# Setup
!pip install -q openai python-dotenv

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## 1. Define Assistant Tools

Create function definitions that give the LLM specific capabilities for BA/QA tasks.


In [ ]:
# Define tools the assistant can use
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'analyze_requirement',
            'description': 'Analyze a requirement for quality issues including ambiguity, testability, and completeness',
            'parameters': {
                'type': 'object',
                'properties': {
                    'requirement_text': {'type': 'string', 'description': 'The requirement text to analyze'},
                    'check_types': {
                        'type': 'array',
                        'items': {'type': 'string', 'enum': ['ambiguity', 'completeness', 'testability', 'consistency']},
                        'description': 'Types of quality checks to perform'
                    }
                },
                'required': ['requirement_text']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'generate_test_cases',
            'description': 'Generate test cases from a requirement or user story',
            'parameters': {
                'type': 'object',
                'properties': {
                    'requirement': {'type': 'string', 'description': 'The requirement to generate tests for'},
                    'test_types': {
                        'type': 'array',
                        'items': {'type': 'string', 'enum': ['positive', 'negative', 'boundary', 'edge_case', 'security']},
                        'description': 'Types of test cases to generate'
                    },
                    'count': {'type': 'integer', 'description': 'Number of test cases to generate', 'default': 5}
                },
                'required': ['requirement']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'create_user_story',
            'description': 'Create a user story with acceptance criteria from a feature description',
            'parameters': {
                'type': 'object',
                'properties': {
                    'feature': {'type': 'string', 'description': 'The feature to write a story for'},
                    'persona': {'type': 'string', 'description': 'The user persona/role'},
                    'context': {'type': 'string', 'description': 'Project or domain context'}
                },
                'required': ['feature']
            }
        }
    }
]

print(f'Defined {len(tools)} assistant tools:')
for t in tools:
    print(f"  - {t['function']['name']}: {t['function']['description'][:60]}...")

## 2. Implement Tool Functions

Create the actual implementations that are called when the LLM invokes a tool.


In [ ]:
def analyze_requirement(requirement_text: str, check_types: list = None) -> dict:
    if not check_types:
        check_types = ['ambiguity', 'completeness', 'testability', 'consistency']
    prompt = f"""Analyze this requirement for quality issues.
Check for: {', '.join(check_types)}

Requirement: {requirement_text}

Return JSON with: quality_score (1-10), issues (array of {{type, description, severity}}), 
improved_version (rewritten requirement). Return ONLY valid JSON."""
    response = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': prompt}], temperature=0.2
    )
    return json.loads(response.choices[0].message.content)

def generate_test_cases(requirement: str, test_types: list = None, count: int = 5) -> list:
    if not test_types:
        test_types = ['positive', 'negative', 'boundary']
    prompt = f"""Generate {count} test cases for this requirement.
Types: {', '.join(test_types)}

Requirement: {requirement}

Return JSON array with: tc_id, title, type, steps, expected_result, priority.
Return ONLY valid JSON."""
    response = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': prompt}], temperature=0.2
    )
    return json.loads(response.choices[0].message.content)

def create_user_story(feature: str, persona: str = 'user', context: str = '') -> dict:
    prompt = f"""Create a user story for: {feature}
Persona: {persona}
Context: {context}

Return JSON with: story_id, title, as_a, i_want, so_that, 
acceptance_criteria (Given/When/Then list), story_points, priority.
Return ONLY valid JSON."""
    response = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': prompt}], temperature=0.3
    )
    return json.loads(response.choices[0].message.content)

# Map function names to implementations
tool_functions = {
    'analyze_requirement': analyze_requirement,
    'generate_test_cases': generate_test_cases,
    'create_user_story': create_user_story
}

print('Tool implementations ready.')

## 3. Conversational Assistant with Tool Calling

Build the assistant loop that handles natural language requests and invokes tools as needed.


In [ ]:
def run_assistant(user_message: str, conversation_history: list = None) -> str:
    """Run the BA/QA assistant with tool calling."""
    if conversation_history is None:
        conversation_history = []
    
    messages = [
        {'role': 'system', 'content': 'You are an expert BA/QA assistant. Help users with requirements analysis, test case generation, and user story creation. Use the available tools when appropriate. Explain your reasoning and provide actionable insights.'},
        *conversation_history,
        {'role': 'user', 'content': user_message}
    ]
    
    response = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools, temperature=0.3
    )
    
    msg = response.choices[0].message
    
    # If the model wants to call tools
    if msg.tool_calls:
        messages.append(msg)
        for tool_call in msg.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            print(f'  [Tool Call] {fn_name}({json.dumps(fn_args)[:80]}...)')
            
            result = tool_functions[fn_name](**fn_args)
            messages.append({
                'role': 'tool',
                'tool_call_id': tool_call.id,
                'content': json.dumps(result)
            })
        
        # Get final response
        final = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=0.3
        )
        return final.choices[0].message.content
    
    return msg.content

# Test the assistant with different requests
test_requests = [
    'Can you analyze this requirement? "The system should handle all user inputs efficiently and display results."',
    'Generate test cases for a password reset feature that sends a reset link via email, valid for 24 hours.',
    'Create a user story for adding a dark mode toggle to our mobile banking app.',
]

for request in test_requests:
    print(f'\nUser: {request}')
    print(f'\nAssistant: {run_assistant(request)}')
    print('\n' + '='*60)

## Exercises
1. Add a `review_brd` tool that checks a BRD document for completeness against a template.
2. Implement conversation memory so the assistant can reference previous interactions.
3. Add a `search_knowledge_base` tool that integrates the RAG system from Chapter 13.
4. Deploy the assistant as a simple web chat interface using Gradio or Streamlit.
